In [ ]:
import requests
import pandas as pd
import time
import plotly.express as px

In [ ]:
# Trocar Futuramente por consulta na Wiki + snake case format
bond_mods_list = [
    "aerial_bond",
    "astral_bond",
    "contagious_bond",
    "covert_bond",
    "duplex_bond",
    "manifold_bond",
    "momentous_bond",
    "mystic_bond",
    "reinforced_bond",
    "restorative_bond",
    "seismic_bond",
    "tandem_bond",
    "tenacious_bond",
    "vicious_bond"
]  

In [ ]:
def get_order_request(url):
    time.sleep(1)
    response = requests.get(url)
    response.raise_for_status()
    return response

def get_bond_mod_offers(bond_mods_list: list[str]):
    df_concat = pd.DataFrame()
    for mod_name in bond_mods_list:
        print(f"Collecting orders from {mod_name}")
        url = f"https://api.warframe.market/v2/orders/item/{mod_name}"
        mod_orders = get_order_request(url)
        df_order = pd.DataFrame(mod_orders.json()["data"])
        df_concat = pd.concat([df_concat, df_order])
    df_concat['user'] = df_concat['user'].str['ingameName']
    df_concat['mod_name'] = mod_name
    df_concat = df_concat[['user', 'rank', 'platinum', 'type', 'quantity', 'perTrade', 'createdAt', 'updatedAt']]
    df_concat = df_concat.sort_values('rank', ascending=False)
    return df_concat

def get_bond_mod_timeseries(bond_mods_list:list[str]):
    collect_modes = ["statistics_closed", "statistics_live"]
    df_concat = pd.DataFrame()
    for mod_name in bond_mods_list:
        print(f"Collecting timeseries from {mod_name}")
        url = f"https://api.warframe.market/v1/items/{mod_name}/statistics"
        mod_orders = get_order_request(url)
        for mode in collect_modes:
            df_order = pd.DataFrame(mod_orders.json()["payload"][mode]["90days"])
            df_order["mode"] = ("live" if mode == "statistics_live" else "closed")
            df_order["mod_name"] = mod_name
            df_concat = pd.concat([df_concat, df_order])
    df_concat = df_concat.sort_values('datetime', ascending=False)
    return df_concat

def clean_timeseries_data(timeseries_df):
    timeseries_df = timeseries_df.copy()
    timeseries_df["datetime"] = pd.to_datetime(timeseries_df["datetime"])
    print(f"size before the filter {timeseries_df.shape[0]}")
    timeseries_df = timeseries_df[
        (timeseries_df["mode"] == "live")
    ]
    print(f"size after the filter {timeseries_df.shape[0]}")
    timeseries_df = (
        timeseries_df
        .groupby(["datetime", "mod_name", "mod_rank", "order_type"], as_index=False)
        .agg(
            min_price=("min_price", "mean"),
            avg_price=("avg_price", "mean"),
            max_price=("max_price", "mean"),
            volume=("volume", "sum"),
        )
        .sort_values(["mod_name", "datetime", "mod_rank", "order_type"])
    )
    print(f"size after the aggregation {timeseries_df.shape[0]}")
    timeseries_df["datetime"] = timeseries_df["datetime"].dt.date
    return timeseries_df

In [ ]:
orders_df = get_bond_mod_offers(bond_mods_list)
display(orders_df)

In [ ]:
orders_df.to_csv('../data/bond_orders.csv', sep=';')

In [ ]:
timeseries_df = get_bond_mod_timeseries(bond_mods_list)
display(timeseries_df)

In [ ]:
timeseries_df.to_csv('../data/bond_90days_orders.csv', sep=';')

In [ ]:
timeseries_df = clean_timeseries_data(timeseries_df)
display(timeseries_df)

size before the filter 6796
size after the filter 4491
size after the aggregation 4491


,datetime,mod_name,mod_rank,order_type,min_price,avg_price,max_price,volume
0,2026-06-30,aerial_bond,0,buy,5.0,10.0,15.0,26
1,2026-06-30,aerial_bond,0,sell,17.0,18.5,20.0,1158
2,2026-06-30,aerial_bond,5,buy,6.0,6.0,6.0,1
3,2026-06-30,aerial_bond,5,sell,18.0,31.5,45.0,147
54,2026-07-01,aerial_bond,0,buy,5.0,10.0,15.0,31
...,...,...,...,...,...,...,...,...
4440,2026-09-25,vicious_bond,0,sell,9.0,14.0,19.0,310
4441,2026-09-25,vicious_bond,5,sell,14.0,20.0,26.0,212
4488,2026-09-26,vicious_bond,0,buy,2.0,3.5,5.0,18
4489,2026-09-26,vicious_bond,0,sell,10.0,15.0,20.0,359


In [51]:
bond_df = timeseries_df[
    timeseries_df["mod_name"].isin(bond_mods_list) &
    (timeseries_df["mod_rank"] == 0)
].copy()

fig = px.line(
    bond_df,
    x="datetime",
    y="avg_price",
    color="mod_name",
    line_dash="order_type",
    markers=True,
    title="Bond Mods — Rank 0 — Buy vs Sell",
    labels={
        "datetime": "Date",
        "avg_price": "Average Price",
        "mod_name": "Mod",
        "order_type": "Order Type"
    }
)

fig.show()